# Heat Pump Design

This notebook explores a minimal heat pump model for Utrecht with annual demand, seasonal COP, monthly and daily distribution, and simple cost/CO2 outputs.

In [8]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from home_energy.domain import HeatPumpConfig
from home_energy.heat_pump import (
    build_heat_pump_daily_profile_figure,
    build_heat_pump_energy_source_breakdown_figure,
    build_heat_pump_monthly_figure,
    heat_pump_explainability_summary,
    heat_pump_validation_warnings,
    simulate_heat_pump,
    simulate_heat_pump_interaction,
)

In [9]:
config = HeatPumpConfig(
    enabled=True,
    annual_heat_demand_kwh=8000.0,
    seasonal_cop=4.0,
    hot_water_included=True,
)

result = simulate_heat_pump(config, year=2026)
result.summary_rows()

[{'metric': 'Annual heat demand', 'value': 8000.0, 'unit': 'kWh heat'},
 {'metric': 'Annual electricity consumption',
  'value': 2000.0,
  'unit': 'kWh electric'},
 {'metric': 'Average COP', 'value': 4.0, 'unit': '-'}]

In [10]:
warnings = heat_pump_validation_warnings(config, household_consumption_kwh=3500.0)
for warning in warnings:
    print(f'WARNING: {warning}')

In [11]:
monthly_figure = build_heat_pump_monthly_figure(result)
monthly_figure

In [12]:
hourly_price = [0.22 if 11 <= hour <= 15 else 0.34 for hour in range(24)]
dynamic_price = [hourly_price[ts.hour] for ts in result.timestamps]

interaction = simulate_heat_pump_interaction(
    result,
    dynamic_price_eur_per_kwh=dynamic_price,
    grid_co2_kg_per_kwh=0.35,
)

source_breakdown_figure = build_heat_pump_energy_source_breakdown_figure(interaction)
daily_profile_figure = build_heat_pump_daily_profile_figure(interaction)
interaction.summary_rows()

[{'metric': 'Solar supplied', 'value': 0.0, 'unit': 'kWh'},
 {'metric': 'Battery supplied', 'value': 0.0, 'unit': 'kWh'},
 {'metric': 'Grid supplied', 'value': 2000.0, 'unit': 'kWh'},
 {'metric': 'Annual operating cost',
  'value': 632.6380368098169,
  'unit': 'EUR'},
 {'metric': 'Annual CO2', 'value': 700.0, 'unit': 'kg'}]

In [13]:
source_breakdown_figure
daily_profile_figure
heat_pump_explainability_summary(interaction)

'The heat pump consumes approximately 2000 kWh of electricity per year to provide 8000 kWh of heat (average COP 4.0). Around 0% is supplied by solar, 0% by battery, and 100% by grid.'